# **Interpreting a CATE estimate**

Practice for the module [“Heterogeneous Effects”](https://ai-interpretability.school), lesson “Interpreting a CATE estimate”.

An estimate $\hat\tau(x)$ is itself a model, and it needs explaining too: along which features the effect differs, where the subgroup borders lie, how confident we are. The lesson “Interpreting a CATE estimate” covers four tools: the causal forest's feature importance, the linear projection of doubly robust scores on the features (BLP on features), Shapley values for the effect model and the interpreter tree. Here we apply them to one and the same estimate and check three claims of the lesson:

1. the importance of a feature for the outcome and its importance for the effect are different quantities;
2. the four tools answer different questions and disagree when features are correlated;
3. explaining the effect model does not test whether there is any heterogeneity at all: SHAP and the interpreter tree decompose noise too.

Notebook plan:

1. synthetic data with a known $\tau(x)$ and a causal forest;
2. importance for the outcome versus importance for the effect;
3. BLP on features;
4. Shapley values for the effect model;
5. the interpreter tree;
6. where it breaks: correlated features and an effect without heterogeneity;
7. application: the growth mindset program in schools;
8. implementations.

The notebook runs on a CPU; the whole run takes two to three minutes.

In [ ]:
!pip install -q econml==0.17.0

In [ ]:
import logging
import warnings
warnings.filterwarnings('ignore', message='IProgress not found')   # shap looks for widgets on import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import shap
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.inspection import permutation_importance
from econml.dml import CausalForestDML
from econml.cate_interpreter import SingleTreeCateInterpreter

warnings.filterwarnings('ignore', message='.*encountered in matmul')   # spurious numpy warnings on some processors
logging.getLogger('shap').setLevel(logging.ERROR)                     # shap logs the background subsampling
SEED = 0
pd.set_option('display.width', 140)

## 1. Data and the causal forest

A randomized experiment on 4000 objects with six features $X\sim\mathcal N(0, I)$, except $x_4$: it is correlated with $x_0$ (correlation coefficient $0.9$). A random half receives the treatment.

- Effect: $\tau(x)=1+1.5\cdot\mathbb 1[x_0>0]+0.5\,x_3$ — a step in $x_0$ and a slope in $x_3$.
- Outcome without treatment: $\mu_0(x)=3x_1+2x_2^2+x_3$. The features $x_1$ and $x_2$ strongly affect the outcome and do not affect the effect at all; $x_3$ affects both.
- $x_4$ affects neither the outcome nor the effect but resembles $x_0$; $x_5$ is pure noise.
- $Y=\mu_0(X)+T\,\tau(X)+\varepsilon$, $\varepsilon\sim\mathcal N(0,1)$.

Half of the objects train the forest, the other half is used for checking and for all explanations.

In [ ]:
def simulate(n, r, heterogeneous=True):
    Xs = r.normal(size=(n, 6))
    Xs[:, 4] = 0.9 * Xs[:, 0] + np.sqrt(1 - 0.9 ** 2) * r.normal(size=n)   # x4 resembles x0
    ts = 1 + 1.5 * (Xs[:, 0] > 0) + 0.5 * Xs[:, 3] if heterogeneous else np.ones(n)
    Ts = r.binomial(1, 0.5, n)
    Ys = 3 * Xs[:, 1] + 2 * Xs[:, 2] ** 2 + Xs[:, 3] + Ts * ts + r.normal(size=n)
    return Xs, Ts, Ys, ts


def gb():
    return GradientBoostingRegressor(n_estimators=200, max_depth=3, random_state=SEED)


def forest(Xa, Ta, Ya):
    return CausalForestDML(model_y=gb(), model_t=LogisticRegression(), discrete_treatment=True,
                           n_estimators=500, min_samples_leaf=20, random_state=SEED).fit(Ya, Ta, X=Xa)


names = [f'x{i}' for i in range(6)]
X, T, Y, tau = simulate(4000, np.random.default_rng(SEED))
tr, te = np.arange(2000), np.arange(2000, 4000)
cf = forest(X[tr], T[tr], Y[tr])
tau_hat = cf.effect(X[te])
print('RMSE of the CATE estimate on the held-out half:', round(np.sqrt(np.mean((tau_hat - tau[te]) ** 2)), 3))
print('correlation of x0 and x4:', round(np.corrcoef(X[:, 0], X[:, 4])[0, 1], 3))

## 2. Importance for the outcome versus importance for the effect

The first trap of the lesson “How to read the estimate and where to go wrong”: a feature that matters for predicting the outcome need not separate the effect. Let us compare the two importances on the same data.

- **Importance for the outcome** — permutation importance of an outcome model: boosting on the features and the treatment, the drop in $R^2$ on the held-out half when the feature is shuffled.
- **Importance for the effect** — `feature_importances_` of the causal forest. In EconML this is not a split frequency: every split on a feature adds to its importance how strongly it separates the effect estimates of the two child nodes, with weight $1/(1+\text{depth})^2$ and only down to depth 4; then the importances are normalized to sum to 1.

**Task 1. Fit an outcome model `gb()` on `np.c_[X[tr], T[tr]]` and `Y[tr]` and compute `permutation_importance` on the held-out half (`n_repeats=5`, `random_state=SEED`). Print a table with two columns: importance for the outcome and `cf.feature_importances_`. What share of the forest's normalized importance falls on $x_0$?**

In [ ]:
# Your code here

**Your answer:**

## 3. BLP on features

The forest's importance is a number without a sign and without an interval. A linear projection gives both. For every held-out object we compute a doubly robust effect score

$$\hat\Gamma_i=\hat\mu_1(X_i)-\hat\mu_0(X_i)+\frac{T_i\,(Y_i-\hat\mu_1(X_i))}{e}-\frac{(1-T_i)\,(Y_i-\hat\mu_0(X_i))}{1-e}$$

with outcome models fitted on the other folds of the held-out half; the propensity in the experiment is known, $e=0.5$. Then we regress $\hat\Gamma_i$ on the features by least squares: $\hat\Gamma_i\approx\beta_0+\sum_j\beta_jX_{ij}$. The coefficient $\beta_j$ is the slope of the best linear approximation to $\tau(x)$ along feature $x_j$ with the others fixed, and a heteroskedasticity-robust standard error (HC3) gives an interval. This is BLP on features ([Semenova and Chernozhukov, 2017](https://arxiv.org/abs/1702.06240)); in the R package `grf` it is the function `best_linear_projection()`.

Note: the projection describes $\tau(x)$, not the forest model. The forest is needed here only for comparison with the other tools.

In [ ]:
def aipw_scores(Xa, Ta, Ya, e=0.5):
    # doubly robust scores with cross-fitting; the propensity is known
    G = np.zeros(len(Ya))
    for a, b in KFold(5, shuffle=True, random_state=SEED).split(Xa):
        m1 = gb().fit(Xa[a][Ta[a] == 1], Ya[a][Ta[a] == 1]).predict(Xa[b])
        m0 = gb().fit(Xa[a][Ta[a] == 0], Ya[a][Ta[a] == 0]).predict(Xa[b])
        G[b] = m1 - m0 + Ta[b] * (Ya[b] - m1) / e - (1 - Ta[b]) * (Ya[b] - m0) / (1 - e)
    return G


G_te = aipw_scores(X[te], T[te], Y[te])
print('mean of the Gamma scores (ATE estimate):', round(G_te.mean(), 3), '| true average effect:', round(tau[te].mean(), 3))

**Task 2. Regress `G_te` on the held-out features with a constant (`sm.OLS(..., sm.add_constant(...)).fit(cov_type='HC3')`), label the rows with the feature names and print the coefficients, standard errors and p-values. What is the coefficient of $x_3$, and how does it compare with the true slope $0.5$? How does the projection share the step in $x_0$ between $x_0$ and the similar $x_4$?**

In [ ]:
# Your code here

**Your answer:**

## 4. Shapley values for the effect model

SHAP decomposes the prediction $\hat\tau(x)$ of one object into feature contributions relative to the mean prediction over a background: $\hat\tau(x)=\phi_0+\sum_j\phi_j(x)$. It is the same decomposition as in the lesson “Basics of the SHAP family of methods” of the module “Classical Methods”, except that the model is an effect estimate rather than an outcome prediction. The contribution $\phi_j(x)$ answers “how much does feature $x_j$ of this object shift the *effect estimate*”, and says nothing about whether the estimate itself is right.

`cf.shap_values(X)` in EconML builds a `shap.Explainer` over the effect model with the masker `shap.maskers.Independent` and a background of 100 objects taken from the given `X`: the values are interventional, and masked features are drawn from the background independently of each other.

**Task 3. Compute `cf.shap_values(X[te][:300], feature_names=names)` and extract the `shap.Explanation` object (a dict of dicts: by outcome, then by treatment). Print the mean absolute contribution $\overline{|\phi_j|}$ over the 300 objects for every feature and draw `shap.plots.beeswarm`. What is the mean absolute contribution of $x_0$ and of $x_4$?**

In [ ]:
# Your code here

**Your answer:**

## 5. The interpreter tree

`SingleTreeCateInterpreter` fits an ordinary `sklearn` regression tree to predict the forest's estimate $\hat\tau(x)$ from the features — that is, it builds a surrogate of the effect model — the same idea as the surrogate model in the lesson “LIME: Local Interpretable Model-agnostic Explanations” of the module “Classical Methods”, only with one surrogate for the whole sample. The tree's leaves are subgroups that are easy to name in words; the number in a leaf is the mean effect estimate of the forest, not the mean actual effect. The tree explains the model, not the data: its splits cannot be read as verified subgroups.

**Task 4. Build `SingleTreeCateInterpreter(max_depth=2, min_samples_leaf=100)` from `cf` and `X[te]`, draw it with `plot(feature_names=names)` and print the feature and threshold of the first split from `tree_model_.tree_`. On which feature and at which threshold does the first split happen?**

In [ ]:
# Your code here

**Your answer:**

## 6. Where it breaks

### 6.1. Correlated features

Let us put into one table what the four tools say about $x_0$ and $x_4$. The feature $x_4$ does not affect the effect but carries almost the same information as $x_0$.

In [ ]:
summary = pd.DataFrame({'forest importance': imp.iloc[:, 1],
                        'BLP: coefficient': blp.params[names],
                        'BLP: p-value': blp.pvalues[names],
                        'SHAP: mean |phi|': mean_abs}).loc[['x0', 'x4', 'x3', 'x1']]
print(summary.round(3))

All three tools give $x_4$ a small but nonzero value, although it does not affect the effect: the forest sometimes splits on $x_4$ when $x_0$ is not in the node's random feature subset, and SHAP with an independent masker repeats what the model did. Only an interval can tell $x_4$ from a cause: for the BLP coefficient of $x_4$ it covers zero. But even BLP helps here only because $x_0$ stands next to it in the regression: remove $x_0$ from the features, and the projection hands the step to $x_4$. None of the tools says “$x_4$ is not a cause here”: that is knowledge about the data, not about the model.

The second row of the same table is about scale. For $x_3$ the forest importance is five times smaller than for $x_0$, while the BLP coefficient is almost the same: forest importance is the share of heterogeneity created by splits, the BLP slope is the change of the effect per unit of the feature. Comparing features by importance and by slope means comparing different quantities.

### 6.2. An effect without heterogeneity

Now the effect is the same for everyone, $\tau(x)=1$. There is no heterogeneity, and any structure in $\hat\tau(x)$ is estimation noise.

**Task 5. Generate data with `simulate(4000, np.random.default_rng(1), heterogeneous=False)`, fit a forest on the first half with `forest`, and repeat three explanations on the second half: the forest importance, the mean absolute SHAP on the first 300 objects and BLP on features. Print the sum of the mean absolute SHAP values over all features, the largest forest importance and the p-value of the joint test of the hypothesis “all feature coefficients are zero” (`f_test` or `wald_test` over the six coefficients; `blp0.f_pvalue` is convenient). Which of the three explanations reports that there is no heterogeneity?**

In [ ]:
# Your code here

**Your answer:**

## 7. Application: the growth mindset program in schools

The data are the same as in the notebook “Causal forest and honest splitting”: a simulation based on the National Study of Learning Mindsets, prepared for the ACIC 2018 workshop and analyzed with a causal forest by [Athey and Wager (2019)](https://arxiv.org/abs/1902.07409). The file `learning_mindset.csv` comes from the repository [Causal Inference for the Brave and True](https://github.com/matheusfacure/python-causality-handbook) by Matheus Facure (MIT license) and is stored in the `data/` folder of the course repository. 10391 students from 76 schools; the treatment `intervention` is a short online program on the idea that abilities can grow; the outcome `achievement_score` is standardized achievement. The treatment is not assigned at random, so outcome and propensity models in `CausalForestDML` are required here; the propensity is estimated, not known.

In [ ]:
URL = 'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/learning_mindset.csv'
lm = pd.read_csv(URL)
Xm = pd.get_dummies(lm.drop(columns=['schoolid', 'intervention', 'achievement_score']),
                    columns=['ethnicity', 'gender', 'school_urbanicity'], drop_first=True).astype(float)
Tm, Ym = lm.intervention.values, lm.achievement_score.values
cfm = CausalForestDML(model_y=gb(), model_t=LogisticRegression(max_iter=1000), discrete_treatment=True,
                      n_estimators=500, min_samples_leaf=50, random_state=SEED).fit(Ym, Tm, X=Xm.values)
cate_m = cfm.effect(Xm.values)
print('students:', len(lm), '| features:', Xm.shape[1], '| mean CATE estimate:', round(cate_m.mean(), 3))

**Task 6. For this estimate print the five features with the largest forest importance and the five features with the largest mean absolute SHAP (over the first 500 students, `cfm.shap_values(Xm.values[:500], feature_names=list(Xm.columns))`), then build an interpreter tree of depth 2 with `min_samples_leaf=500` and print the feature and threshold of the first split. Does the top feature coincide across the three explanations?**

In [ ]:
# Your code here

**Your answer:**

## 8. Implementations

- `econml` (MIT license): `feature_importances_` of `CausalForestDML` and `econml.grf.CausalForest` — heterogeneity created by splits, weighted by depth; `shap_values()` of CATE estimators — a `shap.Explainer` with an independent masker and a background of 100 objects; `econml.cate_interpreter.SingleTreeCateInterpreter` — an `sklearn` tree on the `const_marginal_effect` prediction, with `include_model_uncertainty=True` also intervals in the leaves.
- The R package [grf](https://github.com/grf-labs/grf) (GPL-3.0 license): `variable_importance()` — a depth-weighted split frequency, `best_linear_projection()` — BLP on features from the forest's doubly robust scores.
- [shap](https://github.com/shap/shap) (MIT license): any `shap.plots` chart for the `Explanation` object that EconML returns.

## What to take away

- **Importance for the outcome and for the effect are different numbers.** The features that explain the outcome best may say nothing about the effect, and vice versa.
- **The four tools answer different questions.** Forest importance — where the forest split; BLP — a linear slope of the effect with an interval; SHAP — a decomposition of one object's estimate; the interpreter tree — subgroups that are easy to name.
- **Correlated features share the explanation,** and none of the tools tells a cause from a feature that resembles it.
- **An explanation does not test heterogeneity.** SHAP, forest importance and the tree decompose noise too; the test comes from the BLP interval and from the checks of the lesson “How to read the estimate and where to go wrong”.

Sources:
1. [Wager & Athey, 2018. Estimation and inference of heterogeneous treatment effects using random forests](https://arxiv.org/abs/1510.04342)
2. [Semenova & Chernozhukov, 2017. Debiased machine learning of conditional average treatment effects and other causal functions](https://arxiv.org/abs/1702.06240)
3. [Lundberg & Lee, 2017. A unified approach to interpreting model predictions](https://arxiv.org/abs/1705.07874)
4. [Athey & Wager, 2019. Estimating treatment effects with causal forests: an application](https://arxiv.org/abs/1902.07409)
5. [EconML](https://github.com/py-why/EconML), [shap](https://github.com/shap/shap)